In [57]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰
from sklearn.tree import DecisionTreeRegressor  #回帰木

In [58]:
df = pd.read_csv('datafiles/train.csv')

In [59]:
#欠損値の確認
for c in df.columns:
    null_counts = df[c].isnull().sum()
    if null_counts != 0:
        print(f'{null_counts}  列＝{c}')

259  列＝LotFrontage
1369  列＝Alley
872  列＝MasVnrType
8  列＝MasVnrArea
37  列＝BsmtQual
37  列＝BsmtCond
38  列＝BsmtExposure
37  列＝BsmtFinType1
38  列＝BsmtFinType2
1  列＝Electrical
690  列＝FireplaceQu
81  列＝GarageType
81  列＝GarageYrBlt
81  列＝GarageFinish
81  列＝GarageQual
81  列＝GarageCond
1453  列＝PoolQC
1179  列＝Fence
1406  列＝MiscFeature


In [60]:
#明らかに不要な'id'を除く
df = df.drop(['Id'], axis = 1)

In [61]:
#ダミー変数化する行の抜き出し
to_dummy_cols = []
for c in df.columns:
    if type(c) == str:
        to_dummy_cols.append(c)
print(to_dummy_cols)

['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 'Fen

In [62]:
'''
#strの特徴量の中にNAが混ざっている列
Alley
MasVnrType
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2
Electrical
FireplaceQu
GarageType
GarageFinish
GarageQual
GarageCond
PoolQC
Fence
MiscFeature

#intの特徴量の中にNAが混ざっている列
LotFrontage  NAを0に変更
MasVnrArea   NAを0に変更
GarageYrBlt  NAを0に変更
'''
#data_description.txt を確認すると、すべての特徴量で'NA'に意味があるようだったので補完
to_NA_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 
    'GarageCond', 'Fence', 'MiscFeature'
]
df[to_NA_cols] = df[to_NA_cols].fillna('NA')
df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']] = df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].fillna(0.0)


In [63]:
print(df.columns)

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

In [64]:
#float型に変更
not_to_dummy = ['MSSubClass', 'LotFrontage', 
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 
    'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
    'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice'
]

In [65]:
df[not_to_dummy] = df[not_to_dummy].astype('float64')

In [66]:
#ダミー変数化
to_dummy = set(df.columns) - set(not_to_dummy)
to_dummy = list(to_dummy)
for c in to_dummy:
    dummy = pd.get_dummies(df[c], prefix=c, drop_first = True, dtype = int)
    df = pd.concat([df, dummy], axis = 1)
    df = df.drop([c], axis = 1)

In [67]:
#float型に変更
df = df.astype('float64')

In [68]:
#説明変数と目的変数のデータフレームを作る
df_y = pd.DataFrame(df['SalePrice'])
df_x = df.drop(['SalePrice'], axis = 1)

#標準化
sc_model=StandardScaler()
sc_model.fit(df_x)
sc_x = sc_model.fit_transform(df_x)

In [69]:
#重回帰、リッジ回帰、ラッソ回帰、回帰木を実践し、結果を比較する。
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [70]:
#重回帰
model1 = LinearRegression()
result = cross_validate(model1, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel1のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel1のスコア＝0.5498616628567794


In [71]:
#リッジ回帰
#正則化項の定数を0.01~20まで検証
best_ridgescore = 0
best_alpha = 0

#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    ridgeModel = Ridge(random_state = 0, alpha = i)
    all_result = cross_validate(ridgeModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = i
print(f'正則化項＝{best_alpha}　リッジ回帰のスコア＝{best_ridgescore}')

#完成したリッジ回帰モデルで学習
model2 = Ridge(alpha = best_alpha)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

正則化項＝100　リッジ回帰のスコア＝0.7773772831801897
完成したmodel2のスコア＝0.7773772831801897


In [72]:
#ラッソ回帰
#正則化項の定数を0.01~20まで検証
best_lassoscore = 0
best_alpha = 0
#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    lassoModel = Lasso(random_state = 0, alpha = i)
    all_result = cross_validate(lassoModel, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = i
print(f'正則化項＝{best_alpha}　ラッソ回帰のスコア＝{best_lassoscore}')

#完成したラッソ回帰モデルで学習
model3 = Lasso(alpha = best_alpha)
result = cross_validate(model3, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel3のスコア＝{sum(result['test_score'])/len(result['test_score'])}')


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.589e+11, tolerance: 7.191e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.956e+11, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.658e+11, toleranc

正則化項＝100　ラッソ回帰のスコア＝0.6355578461429475
完成したmodel3のスコア＝0.6355578461429475


In [ ]:
'''
単純なモデルでは、下記のような結果となった。

重回帰のスコア  0.5498616628567794
リッジ回帰のスコア  0.7773772831801897
ラッソ回帰のスコア  0.6355578461429475

以下では、特徴量を改善したモデルでも実験する。
'''

'\n単純なモデルでは、下記のような結果となった。\n\n重回帰のスコア  0.5498616628567776\nリッジ回帰のスコア  0.5838636191087039\nラッソ回帰のスコア  0.5444905254885237\n\n以下では、特徴量を改善したモデルでも実験する。\n'

In [74]:
model2.fit(sc_x, df_y)

,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [75]:
#リッジ回帰の係数と切片の確認
coef_df = pd.DataFrame({
    'col': df_x.columns,
    'coef': model2.coef_
})
print(f'係数: {coef_df }')
print(f'切片: {model2.intercept_}')

係数:                        col          coef
0               MSSubClass  -3991.979995
1              LotFrontage  -1008.982281
2                  LotArea   5162.279218
3              OverallQual  11149.951899
4              OverallCond   5368.292195
5                YearBuilt   4306.153845
6             YearRemodAdd   2212.318403
7               MasVnrArea   4394.453812
8               BsmtFinSF1   3107.394841
9               BsmtFinSF2   1157.970057
10               BsmtUnfSF    -89.306243
11             TotalBsmtSF   3566.444666
12                1stFlrSF   6100.478745
13                2ndFlrSF   8219.109523
14            LowQualFinSF  -1043.109892
15               GrLivArea  11219.307651
16            BsmtFullBath   2098.241294
17            BsmtHalfBath   -116.834625
18                FullBath   4262.127539
19                HalfBath   1900.740275
20            BedroomAbvGr  -1835.621534
21            KitchenAbvGr  -2899.603025
22            TotRmsAbvGrd   5461.404306
23          

In [81]:
coef_df.sort_values('coef', ascending=False)

,col,coef
15,GrLivArea,11219.307651
3,OverallQual,11149.951899
32,PoolArea,8982.824397
237,RoofMatl_WdShngl,8745.629193
13,2ndFlrSF,8219.109523
231,RoofMatl_CompShg,8041.169823
177,Neighborhood_NridgHt,7397.829309
176,Neighborhood_NoRidge,6488.499521
12,1stFlrSF,6100.478745
25,GarageCars,6015.081963


In [77]:
'''
NAが複数行にかけて同一の意味をもつもの
・no basementの意味
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2

・no garageの意味
GarageType
GarageFinish
GarageQual
GarageCond
'''
sc_x = pd.DataFrame(sc_x)
sc_x.columns = df_x.columns
for c in sc_x.columns:
    print(c)


MSSubClass
LotFrontage
LotArea
OverallQual
OverallCond
YearBuilt
YearRemodAdd
MasVnrArea
BsmtFinSF1
BsmtFinSF2
BsmtUnfSF
TotalBsmtSF
1stFlrSF
2ndFlrSF
LowQualFinSF
GrLivArea
BsmtFullBath
BsmtHalfBath
FullBath
HalfBath
BedroomAbvGr
KitchenAbvGr
TotRmsAbvGrd
Fireplaces
GarageYrBlt
GarageCars
GarageArea
WoodDeckSF
OpenPorchSF
EnclosedPorch
3SsnPorch
ScreenPorch
PoolArea
MiscVal
MoSold
YrSold
LandContour_HLS
LandContour_Low
LandContour_Lvl
GarageFinish_NA
GarageFinish_RFn
GarageFinish_Unf
RoofStyle_Gable
RoofStyle_Gambrel
RoofStyle_Hip
RoofStyle_Mansard
RoofStyle_Shed
SaleType_CWD
SaleType_Con
SaleType_ConLD
SaleType_ConLI
SaleType_ConLw
SaleType_New
SaleType_Oth
SaleType_WD
KitchenQual_Fa
KitchenQual_Gd
KitchenQual_TA
Heating_GasA
Heating_GasW
Heating_Grav
Heating_OthW
Heating_Wall
Fence_GdWo
Fence_MnPrv
Fence_MnWw
Fence_NA
Electrical_FuseF
Electrical_FuseP
Electrical_Mix
Electrical_NA
Electrical_SBrkr
MSZoning_FV
MSZoning_RH
MSZoning_RL
MSZoning_RM
ExterQual_Fa
ExterQual_Gd
ExterQual_TA


In [78]:
'''
前のセルの結果、下記のセルが重複するNAのダミー変数化であった。

BsmtQual_NA
BsmtCond_NA
BsmtExposure_NA
BsmtFinType1_NA
BsmtFinType2_NA
　→BsmtQual_NAのみ残し、他は削除

GarageCond_NA
GarageType_NA
GarageQual_NA
　→GarageFinish列で同一の意味を表せるので削除
'''
to_drop = ['BsmtCond_NA', 'BsmtExposure_NA', 'BsmtFinType1_NA', 'BsmtFinType2_NA', 'GarageCond_NA', 'GarageType_NA', 'GarageQual_NA']
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

In [85]:
#改善したデータフレームを用いて、リッジ回帰モデルで学習
result = cross_validate(model1, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel1のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

result = cross_validate(model3, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したmodel3のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel1のスコア＝0.5527321274460048
完成したmodel2のスコア＝0.7774453479883098
完成したmodel3のスコア＝0.6353632468195555


In [84]:
#回帰木
best_score = 0
best_depth = 0
#深さを１～30まで実験
for i in range(1,31):
    model = DecisionTreeRegressor(max_depth = i, random_state = 0)
    all_result = cross_validate(model, sc_x, df_y, cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_score:
        best_score = result
        best_depth = i
print(f'深さ＝{best_depth}　回帰木のスコア＝{best_score}')

model4 = DecisionTreeRegressor(max_depth = best_depth, random_state = 0)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel4のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

深さ＝7　回帰木のスコア＝0.7792099262442557
完成したmodel4のスコア＝0.7774453479883098


In [ ]:
'''
特徴量修正後は下記のような結果となった。単純な特徴量のモデルによる前回の結果からの変化はわずかであった。

完成したmodel1のスコア＝0.5527321274460048
完成したmodel2のスコア＝0.7774453479883098
完成したmodel3のスコア＝0.6353632468195555
完成したmodel4のスコア＝0.7774453479883098

以下では、さらに特徴量を改善したモデルでも実験する。
'''

In [ ]:
#多重共線性の解消

In [ ]:
#クラスタリング